# 03. RabbitMQ Deep Dive - 6가지 메시징 패턴 완전 정복

> **난이도**: 중급 | **예상 소요**: 35분 | **사전 지식**: [01. 프로젝트 개요](./01_project_overview.ipynb) 완료

---

## [목차]
1. [환경 설정](#1.-환경-설정)
2. [AMQP 0-9-1 프로토콜 개요](#2.-AMQP-0-9-1-프로토콜-개요)
3. [connect_robust와 prefetch_count](#3.-connect_robust와-prefetch_count-분석)
4. [Pattern 1: Direct Queue](#4.-Pattern-1:-Direct-Queue) -- Point-to-Point
5. [Pattern 2: Fanout Exchange](#5.-Pattern-2:-Fanout-Exchange) -- Broadcast
6. [Pattern 3: Topic Exchange](#6.-Pattern-3:-Topic-Exchange) -- 라우팅 키 패턴
7. [Pattern 4: DLQ](#7.-Pattern-4:-DLQ) -- Dead Letter Queue
8. [Pattern 5: Priority Queue](#8.-Pattern-5:-Priority-Queue) -- 우선순위
9. [Pattern 6: TTL](#9.-Pattern-6:-TTL) -- 메시지 만료
10. [Management UI 활용법](#10.-RabbitMQ-Management-UI-활용법)
11. [정리 및 요약](#11.-정리-및-요약)

---

## [학습 목표]
- AMQP 0-9-1의 Exchange -> Binding -> Queue 모델 이해
- 6가지 메시징 패턴을 실제 API 호출로 실습
- DLQ로 실패 메시지를 격리하는 방법 체험

> **Tip**: RabbitMQ Management UI(`http://localhost:15672`)를 함께 열어두면 큐의 상태를 시각적으로 확인할 수 있습니다.

> **관련 노트북**: Headers Exchange, Retry+Exponential Backoff는 [07. 고급 패턴](./07_advanced_patterns.ipynb)에서 다룹니다.

---

### [노트북 네비게이션]
| 이전 | 현재 | 다음 |
|------|------|------|
| [02. Redis Deep Dive](./02_redis_deep_dive.ipynb) | **03. RabbitMQ Deep Dive** (현재) | [04. Kafka Deep Dive](./04_kafka_deep_dive.ipynb) |

| # | 패턴 | 핵심 개념 |
|---|------|----------|
| 1 | Direct Queue | Point-to-Point, Default Exchange |
| 2 | Fanout Exchange | Broadcast, 모든 바인딩 큐에 전달 |
| 3 | Topic Exchange | Routing Key 패턴 매칭 (`*`, `#`) |
| 4 | DLQ (Dead Letter Queue) | 실패 메시지 격리, DLX |
| 5 | Priority Queue | 우선순위 기반 소비 순서 |
| 6 | TTL (Time-To-Live) | 메시지 자동 만료 |

## 1. 환경 설정

프로젝트 루트를 `sys.path`에 추가하고, `httpx.AsyncClient`로 FastAPI 엔드포인트를 호출할 준비를 합니다.
모든 API 호출은 비동기(`await`)로 수행합니다.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = str(Path.cwd().parent)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import httpx
BASE_URL = "http://localhost:8000"
client = httpx.AsyncClient(base_url=BASE_URL, timeout=10.0)

서버 헬스체크를 통해 연결 상태를 확인합니다. 응답이 오지 않으면 서버가 기동되지 않은 것입니다.

In [2]:
resp = await client.get("/health")
print(f"Status: {resp.status_code}")
resp.json()

Status: 200


{'api': 'running',
 'redis': 'connected',
 'rabbitmq': 'connected',
 'kafka': 'connected'}

## 2. AMQP 0-9-1 프로토콜 개요

### 왜 AMQP에는 Exchange라는 것이 있을까?

메시지를 보내는 방법을 생각해봅시다. 가장 단순한 방법은 **보내는 사람이 받는 사람에게 직접 전달**하는 것입니다.
Redis Pub/Sub가 바로 이 방식입니다:

\n<div align="center">
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 800 240" width="100%" height="auto"> <rect width="100%" height="100%" fill="#F8FAFC" rx="12" /> <text x="400" y="30" font-family="-apple-system, sans-serif" font-size="14" font-weight="bold" fill="#0F172A" text-anchor="middle">RabbitMQ AMQP 0-9-1 간접 배송 라우팅 모델</text> <g transform="translate(50, 70)"> <rect x="0" y="0" width="130" height="90" rx="8" fill="#E0F2FE" stroke="#0284C7" stroke-width="1.5" /> <text x="65" y="35" font-family="-apple-system, sans-serif" font-size="12" font-weight="bold" fill="#0369A1" text-anchor="middle">Publisher</text> <text x="65" y="55" font-family="-apple-system, sans-serif" font-size="10" fill="#0C4A6E" text-anchor="middle">메시지 발행</text> <text x="65" y="70" font-family="monospace" font-size="8" fill="#0369A1" text-anchor="middle">routing_key="order"</text> </g> <path d="M 180 115 L 245 115" stroke="#0284C7" stroke-width="2" /><polygon points="239,111 245,115 239,119" fill="#0284C7" /> <g transform="translate(250, 70)"> <circle cx="60" cy="45" r="40" fill="#FEF3C7" stroke="#D97706" stroke-width="1.5" /> <text x="60" y="43" font-family="-apple-system, sans-serif" font-size="11" font-weight="bold" fill="#B45309" text-anchor="middle">Exchange</text> <text x="60" y="56" font-family="-apple-system, sans-serif" font-size="8" fill="#78350F" text-anchor="middle">우체국 분류기</text> </g> <path d="M 370 115 L 435 95" stroke="#D97706" stroke-width="2" stroke-dasharray="3 3" /><polygon points="429,91 435,95 429,98" fill="#D97706" /> <text x="402" y="85" font-family="-apple-system, sans-serif" font-size="8" fill="#B45309" text-anchor="middle">Binding 1</text> <path d="M 370 115 L 435 135" stroke="#D97706" stroke-width="2" stroke-dasharray="3 3" /><polygon points="429,132 435,135 429,139" fill="#D97706" /> <text x="402" y="150" font-family="-apple-system, sans-serif" font-size="8" fill="#B45309" text-anchor="middle">Binding 2</text> <g transform="translate(450, 50)"> <rect x="0" y="0" width="130" height="50" rx="6" fill="#DCFCE7" stroke="#15803D" stroke-width="1.5" /> <text x="65" y="24" font-family="-apple-system, sans-serif" font-size="11" font-weight="bold" fill="#166534" text-anchor="middle">Queue A (order)</text> <text x="65" y="38" font-family="-apple-system, sans-serif" font-size="9" fill="#14532D" text-anchor="middle">보관창고 1</text> </g> <g transform="translate(450, 130)"> <rect x="0" y="0" width="130" height="50" rx="6" fill="#DCFCE7" stroke="#15803D" stroke-width="1.5" /> <text x="65" y="24" font-family="-apple-system, sans-serif" font-size="11" font-weight="bold" fill="#166534" text-anchor="middle">Queue B (other)</text> <text x="65" y="38" font-family="-apple-system, sans-serif" font-size="9" fill="#14532D" text-anchor="middle">보관창고 2</text> </g> <path d="M 580 75 L 635 75" stroke="#15803D" stroke-width="2" /><polygon points="629,71 635,75 629,79" fill="#15803D" /> <g transform="translate(640, 50)"> <rect x="0" y="0" width="110" height="50" rx="6" fill="#F3E8FF" stroke="#7E22CE" stroke-width="1.5" /> <text x="55" y="24" font-family="-apple-system, sans-serif" font-size="11" font-weight="bold" fill="#6B21A8" text-anchor="middle">Consumer A</text> <text x="55" y="38" font-family="-apple-system, sans-serif" font-size="9" fill="#581C87" text-anchor="middle">구독 수신자</text> </g> </svg>
</div>\n

## 3. `connect_robust`와 `prefetch_count` 분석

우리 프로젝트의 `RabbitMQBroker.connect()` 메서드를 살펴봅니다.

```python
# app/brokers/rabbitmq_broker.py
async def connect(self) -> None:
    self.connection = await aio_pika.connect_robust(settings.rabbitmq_url)
    self.channel = await self.connection.channel()
    await self.channel.set_qos(prefetch_count=10)
```

### `connect_robust` vs `connect`

| 항목 | `connect` | `connect_robust` |
|------|-----------|-------------------|
| 자동 재연결 | X | O |
| Exchange/Queue 재선언 | X | O |
| 프로덕션 권장 | X | **O** |

### `prefetch_count=10`의 의미

Consumer가 한 번에 최대 **10개**의 unacked 메시지를 받을 수 있습니다.  
이 값이 너무 크면 특정 Consumer에 메시지가 몰리고, 너무 작으면 처리량이 떨어집니다.

```text
RabbitMQ Queue:  [msg1][msg2][msg3][msg4]...[msg10][msg11][msg12]...
                  \________________________/        ↑
                   Consumer에게 전달됨 (10개)       대기 중 (prefetch 한도)
```


### Competing Consumers 패턴 (경쟁적 소비)

`prefetch_count`는 **Competing Consumers** 패턴의 핵심입니다.
여러 Consumer가 같은 큐에서 메시지를 경쟁적으로 가져가는 구조입니다.

\n<div align="center">
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 800 220" width="100%" height="auto"> <rect width="100%" height="100%" fill="#F8FAFC" rx="12" /> <text x="400" y="30" font-family="-apple-system, sans-serif" font-size="14" font-weight="bold" fill="#0F172A" text-anchor="middle">prefetch_count와 Competing Consumers (Fair Dispatch)</text> <g transform="translate(40, 80)"> <rect x="0" y="0" width="220" height="60" rx="6" fill="#DCFCE7" stroke="#15803D" stroke-width="1.5" /> <text x="110" y="22" font-family="-apple-system, sans-serif" font-size="11" font-weight="bold" fill="#166534" text-anchor="middle">Queue: [Job 4] [Job 3] [Job 2] [Job 1]</text> <text x="110" y="42" font-family="-apple-system, sans-serif" font-size="9" fill="#14532D" text-anchor="middle">대기 중인 무거운 처리 작업들</text> </g> <path d="M 260 110 L 390 75" stroke="#64748B" stroke-width="2" /><polygon points="384,72 390,75 385,80" fill="#64748B" /> <path d="M 260 110 L 390 145" stroke="#64748B" stroke-width="2" /><polygon points="385,140 390,145 384,148" fill="#64748B" /> <g transform="translate(410, 40)"> <rect x="0" y="0" width="340" height="60" rx="6" fill="#F3E8FF" stroke="#7E22CE" stroke-width="1.5" /> <text x="10" y="22" font-family="-apple-system, sans-serif" font-size="11" font-weight="bold" fill="#6B21A8">Consumer A (prefetch_count = 1)</text> <text x="10" y="40" font-family="-apple-system, sans-serif" font-size="9" fill="#581C87">⚡ 느린 워커: 현재 [Job 1] 처리 중 (끝날 때까지 대기)</text> </g> <g transform="translate(410, 120)"> <rect x="0" y="0" width="340" height="60" rx="6" fill="#F3E8FF" stroke="#7E22CE" stroke-width="1.5" /> <text x="10" y="22" font-family="-apple-system, sans-serif" font-size="11" font-weight="bold" fill="#6B21A8">Consumer B (prefetch_count = 1)</text> <text x="10" y="40" font-family="-apple-system, sans-serif" font-size="9" fill="#581C87">🚀 빠른 워커: [Job 2] 처리완료 후 바로 [Job 3], [Job 4] 수신</text> </g> <rect x="180" y="170" width="440" height="24" rx="4" fill="#F1F5F9" stroke="#CBD5E1" stroke-width="1" /> <text x="400" y="186" font-family="-apple-system, sans-serif" font-size="10" font-weight="bold" fill="#475569" text-anchor="middle">공평 분배 (Fair Dispatch) — 무조건 Round-robin하지 않고 여유있는 워커가 가져감</text> </svg>
</div>\n

---
## 4. Pattern 1: Direct Queue (Point-to-Point)

### 택배 직접 배송으로 이해하기

Direct Queue는 **"이름으로 직접 찾아가는 택배"**와 같습니다.

보내는 사람이 택배 상자에 **받는 사람 이름**(= 큐 이름)을 적으면,
택배 기사(Default Exchange)가 **그 이름의 집(Queue)에 직접 배달**합니다.
별도의 분류 규칙(Binding)을 설정할 필요가 없습니다.

\n<div align="center">
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 800 220" width="100%" height="auto"> <rect width="100%" height="100%" fill="#F8FAFC" rx="12" /> <text x="400" y="30" font-family="-apple-system, sans-serif" font-size="14" font-weight="bold" fill="#0F172A" text-anchor="middle">Pattern 1: Direct Exchange (정확한 매칭 라우팅)</text> <g transform="translate(60, 80)"> <rect x="0" y="0" width="120" height="60" rx="6" fill="#E0F2FE" stroke="#0284C7" stroke-width="1.5" /> <text x="60" y="26" font-family="-apple-system, sans-serif" font-size="11" font-weight="bold" fill="#0369A1" text-anchor="middle">Msg 발행</text> <text x="60" y="44" font-family="monospace" font-size="9" fill="#0C4A6E" text-anchor="middle">key: "delivery"</text> </g> <path d="M 180 110 L 255 110" stroke="#0284C7" stroke-width="2.5" /><polygon points="249,106 255,110 249,114" fill="#0284C7" /> <g transform="translate(260, 70)"> <circle cx="40" cy="40" r="35" fill="#FEF3C7" stroke="#D97706" stroke-width="1.5" /> <text x="40" y="45" font-family="-apple-system, sans-serif" font-size="10" font-weight="bold" fill="#B45309" text-anchor="middle">Direct EX</text> </g> <path d="M 340 110 L 445 70" stroke="#D97706" stroke-width="2" /><polygon points="439,67 445,70 440,75" fill="#D97706" /> <text x="390" y="80" font-family="monospace" font-size="8" fill="#B45309" text-anchor="middle">key=="delivery"</text> <path d="M 340 110 L 445 150" stroke="#94A3B8" stroke-width="1.5" stroke-dasharray="4 4" /> <text x="390" y="145" font-family="monospace" font-size="8" fill="#64748B" text-anchor="middle">key=="sms" (불일치)</text> <g transform="translate(460, 40)"> <rect x="0" y="0" width="160" height="50" rx="6" fill="#DCFCE7" stroke="#15803D" stroke-width="1.5" /> <text x="80" y="24" font-family="-apple-system, sans-serif" font-size="11" font-weight="bold" fill="#166534" text-anchor="middle">Queue: delivery</text> <text x="80" y="38" font-family="-apple-system, sans-serif" font-size="9" fill="#14532D" text-anchor="middle">수신 성공 (메시지 적재)</text> </g> <g transform="translate(460, 130)"> <rect x="0" y="0" width="160" height="50" rx="6" fill="#F1F5F9" stroke="#94A3B8" stroke-width="1.5" opacity="0.6" /> <text x="80" y="24" font-family="-apple-system, sans-serif" font-size="11" font-weight="bold" fill="#475569" text-anchor="middle">Queue: sms</text> <text x="80" y="38" font-family="-apple-system, sans-serif" font-size="9" fill="#64748B" text-anchor="middle">메시지 받지 못함</text> </g> </svg>
</div>\n

Direct Queue에 메시지를 발행합니다. `POST /rabbitmq/direct/publish` 엔드포인트를 사용합니다.

In [6]:
resp = await client.post(
    "/rabbitmq/direct/publish",
    params={"queue_name": "order-queue"},
    json={"content": "create_order", "metadata": {"item": "laptop", "qty": 2}}
)
print(f"Status: {resp.status_code}")
resp.json()

Status: 200


{'broker': 'rabbitmq',
 'pattern': 'direct',
 'destination': 'test-queue',
 'queue_message_count': 4,
 'elapsed_ms': 21.532}

큐 상태를 조회하여 메시지가 적재되었는지 확인합니다.

In [7]:
resp = await client.get("/rabbitmq/queue/info/order-queue")
print(f"Queue Info: {resp.json()}")

Queue Info: {'queue': 'order-queue', 'error': "NOT_FOUND - no queue 'order-queue' in vhost '/'"}


### Direct Queue 결과 분석

응답에서 확인할 포인트:
- `pattern: "direct"` - Default Exchange를 통한 직접 전달
- `queue_message_count` - 발행 직전의 큐 메시지 수 (발행 후 1 증가 예상)
- Consumer가 없으면 메시지는 큐에 계속 쌓입니다

---
## 5. Pattern 2: Fanout Exchange (Broadcast)

### 교내 방송으로 이해하기

Fanout Exchange는 **학교 교내 방송**과 같습니다.

교장 선생님(Producer)이 방송실(Fanout Exchange)에서 "오늘 체육대회입니다"라고 방송하면,
**모든 교실(Queue)에 동시에 같은 내용이 전달**됩니다.
어떤 교실에 보낼지 선택하는 것이 아니라, **연결된 모든 곳에 복사**됩니다.


<div align="center"> <svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 800 220" width="100%" height="auto"> <rect width="100%" height="100%" fill="#F8FAFC" rx="12" /> <text x="400" y="30" font-family="-apple-system, sans-serif" font-size="14" font-weight="bold" fill="#0F172A" text-anchor="middle">Pattern 2: Fanout Exchange (무조건 브로드캐스트)</text> <g transform="translate(60, 80)"> <rect x="0" y="0" width="120" height="60" rx="6" fill="#E0F2FE" stroke="#0284C7" stroke-width="1.5" /> <text x="60" y="26" font-family="-apple-system, sans-serif" font-size="11" font-weight="bold" fill="#0369A1" text-anchor="middle">Msg 발행</text> <text x="60" y="44" font-family="monospace" font-size="9" fill="#0C4A6E" text-anchor="middle">key 무시됨 ("")</text> </g> <path d="M 180 110 L 255 110" stroke="#0284C7" stroke-width="2.5" /><polygon points="249,106 255,110 249,114" fill="#0284C7" /> <g transform="translate(260, 70)"> <circle cx="40" cy="40" r="35" fill="#FEF3C7" stroke="#D97706" stroke-width="1.5" /> <text x="40" y="45" font-family="-apple-system, sans-serif" font-size="10" font-weight="bold" fill="#B45309" text-anchor="middle">Fanout EX</text> </g> <path d="M 340 110 L 445 70" stroke="#D97706" stroke-width="2" /><polygon points="439,67 445,70 440,75" fill="#D97706" /> <path d="M 340 110 L 445 150" stroke="#D97706" stroke-width="2" /><polygon points="440,145 445,150 439,153" fill="#D97706" /> <g transform="translate(460, 40)"> <rect x="0" y="0" width="180" height="50" rx="6" fill="#DCFCE7" stroke="#15803D" stroke-width="1.5" /> <text x="90" y="24" font-family="-apple-system, sans-serif" font-size="11" font-weight="bold" fill="#166534" text-anchor="middle">Queue: notifications</text> <text x="90" y="38" font-family="-apple-system, sans-serif" font-size="9" fill="#14532D" text-anchor="middle">복제 메시지 수신</text> </g> <g transform="translate(460, 130)"> <rect x="0" y="0" width="180" height="50" rx="6" fill="#DCFCE7" stroke="#15803D" stroke-width="1.5" /> <text x="90" y="24" font-family="-apple-system, sans-serif" font-size="11" font-weight="bold" fill="#166534" text-anchor="middle">Queue: audit-log</text> <text x="90" y="38" font-family="-apple-system, sans-serif" font-size="9" fill="#14532D" text-anchor="middle">복제 메시지 수신</text> </g> </svg> </div>


먼저 Fanout Exchange에 큐를 바인딩합니다. 바인딩 없이 발행하면 메시지는 버려집니다.

In [ ]:
# 2개의 큐를 fanout exchange에 바인딩
for q in ["notifications", "audit-log"]:
    resp = await client.post(
        "/rabbitmq/fanout/bind",
        params={"queue_name": q}
    )
    print(f"Bind {q}: {resp.json()}")

Fanout Exchange에 메시지를 발행합니다. 바인딩된 모든 큐(`notifications`, `audit-log`)에 동일한 메시지가 전달됩니다.

In [ ]:
resp = await client.post(
    "/rabbitmq/fanout/publish",
    json={"content": "user_signup", "metadata": {"user_id": 42}}
)
print(f"Fanout publish: {resp.json()}")

각 큐의 상태를 조회하여 두 큐 모두에 메시지가 도착했는지 확인합니다.

In [ ]:
for q in ["notifications", "audit-log"]:
    resp = await client.get(f"/rabbitmq/queue/info/{q}")
    print(f"{q}: {resp.json()}")

---
## 6. Pattern 3: Topic Exchange (Routing Key 패턴 매칭)

### 신문 구독으로 이해하기

Topic Exchange는 **신문 구독 서비스**와 같습니다.

신문사(Producer)는 기사를 **카테고리 태그**(routing_key)를 붙여서 발행합니다.
예: `sports.soccer.korea`, `sports.baseball.japan`, `economy.stock.korea`

구독자(Consumer)는 자신이 관심 있는 **카테고리 패턴**(binding_key)을 등록합니다.
신문사의 분류 시스템(Topic Exchange)이 패턴에 맞는 구독자에게만 기사를 전달합니다.

```text
[신문 구독 비유]

  신문사가 기사 발행                   분류 시스템               구독자 사서함
  (Producer)                       (Topic Exchange)           (Queue)

  +----------------------+      +------------------+
  | 기사 태그:            |      |                  |     +-------------------+
  | "sports.soccer.korea"| ---> |  구독 패턴에     | --> | 축구팬 사서함      |
  +----------------------+      |  따라 분류       |     | (sports.soccer.*) |
                                |                  |     +-------------------+
                                |                  |
                                |                  |     +-------------------+
                                |                  | --> | 스포츠 종합 사서함 |
                                |                  |     | (sports.#)        |
                                +------------------+     +-------------------+
```

### 패턴 매칭 규칙: `*`와 `#`의 차이

routing_key는 `.`(점)으로 단어를 구분합니다. 예: `sports.soccer.korea` (3개의 단어)

| 와일드카드 | 의미 | 신문 구독 비유 |
|-----------|------|--------------|
| `*` (별표) | **정확히 1개** 단어와 매칭 | "스포츠의 어떤 종목이든, 한국 기사만 보겠다" |
| `#` (샵) | **0개 이상**의 단어와 매칭 | "스포츠라면 뭐든지 다 보겠다" |

### 단계별 매칭 예시

기사(메시지)의 routing_key가 `sports.soccer.korea`일 때, 각 구독 패턴이 매칭되는지 확인해봅시다:

```text
기사 태그(routing_key): sports.soccer.korea
                        [단어1].[단어2].[단어3]

구독 패턴(binding_key)          매칭 여부     이유
---------------------------------------------------------------------------
sports.soccer.korea             [O] 매칭     정확히 일치
sports.soccer.*                 [O] 매칭     * = "korea" (1개 단어 매칭)
sports.*.korea                  [O] 매칭     * = "soccer" (1개 단어 매칭)
sports.#                        [O] 매칭     # = "soccer.korea" (2개 단어 매칭)
#                               [O] 매칭     # = 전체 (모든 메시지 수신)
sports.soccer                   [X] 불일치   단어가 2개뿐 (3개 필요)
sports.baseball.*               [X] 불일치   baseball != soccer
*.soccer                        [X] 불일치   단어가 2개뿐 (3개 필요, *는 1개만 매칭)
```

### 실전 시나리오

```text
[주문 이벤트 라우팅 예시]

  routing_key 구조:  order.{이벤트}.{국가}

  발행되는 메시지들:
    order.created.kr     (한국 주문 생성)
    order.created.us     (미국 주문 생성)
    order.cancelled      (주문 취소 -- 국가 없음, 단어 2개)

  구독 패턴별 수신 결과:
  +---------------------+------------------------------------------+
  | binding_key         | 수신하는 메시지                            |
  +---------------------+------------------------------------------+
  | order.created.*     | order.created.kr, order.created.us       |
  |                     | (생성 이벤트만, 모든 국가)                  |
  +---------------------+------------------------------------------+
  | order.*.kr          | order.created.kr                         |
  |                     | (한국 주문만, 모든 이벤트)                  |
  +---------------------+------------------------------------------+
  | order.#             | order.created.kr, order.created.us,      |
  |                     | order.cancelled  (모든 주문 이벤트)        |
  +---------------------+------------------------------------------+
```

> **Direct vs Topic**: Direct는 routing_key가 **정확히** 일치해야 하지만, Topic은 **패턴**으로 유연하게 매칭합니다.
> 라우팅 조건이 복잡해질수록 Topic Exchange가 유리합니다.

<div align="center"> <svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 800 240" width="100%" height="auto"> <rect width="100%" height="100%" fill="#F8FAFC" rx="12" /> <text x="400" y="30" font-family="-apple-system, sans-serif" font-size="14" font-weight="bold" fill="#0F172A" text-anchor="middle">Pattern 3: Topic Exchange (와일드카드 패턴 매칭)</text> <g transform="translate(40, 90)"> <rect x="0" y="0" width="130" height="60" rx="6" fill="#E0F2FE" stroke="#0284C7" stroke-width="1.5" /> <text x="65" y="26" font-family="-apple-system, sans-serif" font-size="10" font-weight="bold" fill="#0369A1" text-anchor="middle">발행: sports.soccer.ko</text> </g> <path d="M 170 120 L 235 120" stroke="#0284C7" stroke-width="2" /><polygon points="229,116 235,120 229,124" fill="#0284C7" /> <g transform="translate(240, 80)"> <circle cx="40" cy="40" r="35" fill="#FEF3C7" stroke="#D97706" stroke-width="1.5" /> <text x="40" y="45" font-family="-apple-system, sans-serif" font-size="10" font-weight="bold" fill="#B45309" text-anchor="middle">Topic EX</text> </g> <path d="M 320 120 L 425 70" stroke="#D97706" stroke-width="2" /><polygon points="419,67 425,70 420,75" fill="#D97706" /> <text x="365" y="80" font-family="monospace" font-size="9" fill="#B45309" text-anchor="middle">sports.#</text> <path d="M 320 120 L 425 170" stroke="#D97706" stroke-width="2" /><polygon points="420,165 425,170 419,173" fill="#D97706" /> <text x="365" y="160" font-family="monospace" font-size="9" fill="#B45309" text-anchor="middle">*.soccer.*</text> <g transform="translate(440, 40)"> <rect x="0" y="0" width="320" height="60" rx="6" fill="#DCFCE7" stroke="#15803D" stroke-width="1.5" /> <text x="10" y="24" font-family="-apple-system, sans-serif" font-size="11" font-weight="bold" fill="#166534">Queue: sports-all</text> <text x="10" y="42" font-family="-apple-system, sans-serif" font-size="9" fill="#14532D">matched (sports로 시작하는 모든 단어 수신)</text> </g> <g transform="translate(440, 140)"> <rect x="0" y="0" width="320" height="60" rx="6" fill="#DCFCE7" stroke="#15803D" stroke-width="1.5" /> <text x="10" y="24" font-family="-apple-system, sans-serif" font-size="11" font-weight="bold" fill="#166534">Queue: soccer-only</text> <text x="10" y="42" font-family="-apple-system, sans-serif" font-size="9" fill="#14532D">matched (중간에 soccer가 들어가는 3단어 수신)</text> </g> </svg> </div>


Topic Exchange에 서로 다른 패턴의 binding key로 큐를 바인딩합니다.

In [ ]:
# 특정 패턴 바인딩: order.created.* (생성 이벤트만)
resp = await client.post(
    "/rabbitmq/topic/bind",
    params={"queue_name": "order-created-q", "binding_key": "order.created.*"}
)
print(f"Bind created: {resp.json()}")

In [ ]:
# 와일드카드 바인딩: order.# (모든 주문 이벤트)
resp = await client.post(
    "/rabbitmq/topic/bind",
    params={"queue_name": "all-orders-q", "binding_key": "order.#"}
)
print(f"Bind all: {resp.json()}")

`order.created.kr` routing key로 메시지를 발행합니다. 두 큐 모두 매칭되어야 합니다.

In [ ]:
resp = await client.post(
    "/rabbitmq/topic/publish",
    json={"routing_key": "order.created.kr", "content": "new order from KR"}
)
print(f"Topic publish: {resp.json()}")

`order.cancelled` routing key로 발행하면 `order.#`만 매칭되고, `order.created.*`는 매칭되지 않습니다.

In [ ]:
resp = await client.post(
    "/rabbitmq/topic/publish",
    json={"routing_key": "order.cancelled", "content": "order cancelled"}
)
print(f"Topic publish (cancelled): {resp.json()}")

큐 상태를 비교합니다. `order-created-q`는 1개, `all-orders-q`는 2개의 메시지가 있어야 합니다.

In [ ]:
for q in ["order-created-q", "all-orders-q"]:
    resp = await client.get(f"/rabbitmq/queue/info/{q}")
    print(f"{q}: {resp.json()}")

---
## 7. Pattern 4: DLQ (Dead Letter Queue)

### 병원 응급실로 이해하기

DLQ는 **병원의 중환자실 이송 시스템**과 같습니다.

일반 진료실(Main Queue)에서 의사(Consumer)가 환자(메시지)를 치료합니다.
치료에 성공하면 퇴원(ACK)시키고, 치료에 실패하면 중환자실(DLQ)로 이송합니다.
중환자실로 이송하는 규칙을 DLX(Dead Letter Exchange)라고 합니다.


<div align="center"> <svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 800 240" width="100%" height="auto"> <rect width="100%" height="100%" fill="#F8FAFC" rx="12" /> <text x="400" y="30" font-family="-apple-system, sans-serif" font-size="14" font-weight="bold" fill="#0F172A" text-anchor="middle">Pattern 4: DLQ (Dead Letter Queue) 에러 핸들링 아키텍처</text> <g transform="translate(40, 75)"> <rect x="0" y="0" width="150" height="60" rx="6" fill="#DCFCE7" stroke="#15803D" stroke-width="1.5" /> <text x="75" y="26" font-family="-apple-system, sans-serif" font-size="11" font-weight="bold" fill="#166534" text-anchor="middle">Queue: main-queue</text> <text x="75" y="44" font-family="-apple-system, sans-serif" font-size="9" fill="#14532D" text-anchor="middle">메인 처리 대기열</text> </g> <path d="M 190 105 L 245 105" stroke="#15803D" stroke-width="2" /><polygon points="239,101 245,105 239,109" fill="#15803D" /> <g transform="translate(250, 75)"> <rect x="0" y="0" width="140" height="60" rx="6" fill="#FEE2E2" stroke="#B91C1C" stroke-width="1.5" /> <text x="70" y="26" font-family="-apple-system, sans-serif" font-size="11" font-weight="bold" fill="#991B1B" text-anchor="middle">Consumer</text> <text x="70" y="44" font-family="-apple-system, sans-serif" font-size="9" fill="#7F1D1D" text-anchor="middle">에러 발생 / NACK 반환</text> </g> <path d="M 390 105 L 455 105" stroke="#B91C1C" stroke-width="2" /><polygon points="449,101 455,105 449,109" fill="#B91C1C" /> <g transform="translate(460, 75)"> <circle cx="35" cy="30" r="28" fill="#FEE2E2" stroke="#B91C1C" stroke-width="1.5" /> <text x="35" y="34" font-family="-apple-system, sans-serif" font-size="9" font-weight="bold" fill="#991B1B" text-anchor="middle">DLX (Ex)</text> </g> <path d="M 530 105 L 595 105" stroke="#B91C1C" stroke-width="2" /><polygon points="589,101 595,105 589,109" fill="#B91C1C" /> <g transform="translate(600, 75)"> <rect x="0" y="0" width="160" height="60" rx="6" fill="#FEE2E2" stroke="#B91C1C" stroke-width="2" /> <text x="80" y="26" font-family="-apple-system, sans-serif" font-size="11" font-weight="bold" fill="#991B1B" text-anchor="middle">Queue: main-queue.dlq</text> <text x="80" y="44" font-family="-apple-system, sans-serif" font-size="9" fill="#7F1D1D" text-anchor="middle">죽은 메시지 적재 (관리자 분석)</text> </g> </svg> </div>


DLQ 세트를 구성합니다. 메인 큐와 DLQ/DLX가 함께 생성됩니다.

In [ ]:
resp = await client.post("/rabbitmq/dlq/setup", params={"queue_name": "payment-queue"})
print(f"DLQ setup: {resp.json()}")

메인 큐에 메시지를 발행합니다. Consumer가 nack하면 DLQ로 이동하게 됩니다.

In [ ]:
resp = await client.post(
    "/rabbitmq/direct/publish",
    params={"queue_name": "payment-queue"},
    json={"content": "payment", "metadata": {"payment_id": "PAY-001", "amount": 50000}}
)
print(f"Publish to main queue: {resp.json()}")

DLQ에 쌓인 메시지를 조회합니다. Consumer가 처리를 실패한 메시지가 여기에 격리됩니다.

In [ ]:
resp = await client.get("/rabbitmq/dlq/messages", params={"queue_name": "payment-queue"})
print(f"DLQ messages: {resp.json()}")

### DLQ 활용 팁

- DLQ 메시지에는 `x-death` 헤더가 포함되어 실패 원인을 추적할 수 있습니다
- 주기적으로 DLQ를 모니터링하여 시스템 문제를 조기에 발견하세요
- 재처리가 가능한 메시지는 DLQ에서 꺼내 메인 큐로 다시 발행할 수 있습니다

### Manual ACK/NACK 패턴

RabbitMQ의 메시지 처리 보장은 **ACK/NACK** 메커니즘에 기반합니다.

| 동작 | 메서드 | 효과 |
|------|--------|------|
| **ACK** | `message.ack()` | 처리 완료 → 큐에서 삭제 |
| **NACK** | `message.nack(requeue=True)` | 처리 실패 → 큐로 복귀 (재시도) |
| **NACK** | `message.nack(requeue=False)` | 처리 실패 → DLQ로 이동 |
| **Reject** | `message.reject(requeue=False)` | 단일 메시지 거부 → DLQ |

우리 코드에서 `async with message.process():`는 내부적으로:
- 정상 종료 시 → **자동 ACK**
- 예외 발생 시 → **자동 NACK (requeue=True)**

```python
# 프로젝트 코드 (rabbitmq_broker.py:92-95)
async with queue.iterator() as queue_iter:
    async for message in queue_iter:
        async with message.process():  # ← 자동 ACK/NACK
            data = json.loads(message.body.decode())
            await callback(data)
```

**수동 NACK가 필요한 경우**: 비즈니스 로직에 따라 재시도 vs DLQ 결정이 필요할 때
→ `message.process()` 대신 직접 `ack()`/`nack()` 호출

---
## 8. Pattern 5: Priority Queue

### 병원 응급 분류(Triage)로 이해하기

Priority Queue는 **병원 응급실의 환자 분류 시스템(Triage)**과 같습니다.

응급실에 환자가 도착하면, 증상의 심각도에 따라 **우선순위**가 매겨집니다.
대기실(Queue)에 먼저 온 환자라도, 뒤에 더 심각한 환자가 오면 **심각한 환자가 먼저 치료**됩니다.


<div align="center"> <svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 800 200" width="100%" height="auto"> <rect width="100%" height="100%" fill="#F8FAFC" rx="12" /> <text x="400" y="30" font-family="-apple-system, sans-serif" font-size="14" font-weight="bold" fill="#0F172A" text-anchor="middle">Pattern 5: Priority Queue (우선순위 분류기)</text> <g transform="translate(60, 70)"> <text x="60" y="-10" font-family="-apple-system, sans-serif" font-size="10" font-weight="bold" fill="#64748B" text-anchor="middle">무순서 진입</text> <rect x="0" y="0" width="120" height="30" rx="4" fill="#E2E8F0" stroke="#94A3B8" /> <text x="60" y="19" font-family="sans-serif" font-size="10" fill="#334155" text-anchor="middle">A (priority=1)</text> <rect x="0" y="38" width="120" height="30" rx="4" fill="#FCA5A5" stroke="#EF4444" /> <text x="60" y="57" font-family="sans-serif" font-size="10" fill="#991B1B" text-anchor="middle">B (priority=10)</text> <rect x="0" y="76" width="120" height="30" rx="4" fill="#FDE047" stroke="#EAB308" /> <text x="60" y="95" font-family="sans-serif" font-size="10" fill="#854d0e" text-anchor="middle">C (priority=5)</text> </g> <path d="M 200 120 L 265 120" stroke="#64748B" stroke-width="2" /><polygon points="259,116 265,120 259,124" fill="#64748B" /> <g transform="translate(280, 80)"> <rect x="0" y="0" width="280" height="70" rx="6" fill="#DCFCE7" stroke="#15803D" stroke-width="1.5" /> <text x="140" y="-10" font-family="-apple-system, sans-serif" font-size="11" font-weight="bold" fill="#15803D" text-anchor="middle">Priority Queue 내 재배치</text> <rect x="15" y="20" width="70" height="35" rx="4" fill="#FCA5A5" stroke="#EF4444" /><text x="50" y="42" font-family="sans-serif" font-size="9" font-weight="bold" fill="#991B1B" text-anchor="middle">B (10)</text> <rect x="105" y="20" width="70" height="35" rx="4" fill="#FDE047" stroke="#EAB308" /><text x="140" y="42" font-family="sans-serif" font-size="9" font-weight="bold" fill="#854d0e" text-anchor="middle">C (5)</text> <rect x="195" y="20" width="70" height="35" rx="4" fill="#E2E8F0" stroke="#94A3B8" /><text x="230" y="42" font-family="sans-serif" font-size="9" font-weight="bold" fill="#334155" text-anchor="middle">A (1)</text> </g> <path d="M 580 120 L 635 120" stroke="#15803D" stroke-width="2" /><polygon points="629,116 635,120 629,124" fill="#15803D" /> <g transform="translate(650, 90)"> <rect x="0" y="0" width="110" height="50" rx="6" fill="#F3E8FF" stroke="#7E22CE" stroke-width="1.5" /> <text x="55" y="24" font-family="-apple-system, sans-serif" font-size="11" font-weight="bold" fill="#6B21A8" text-anchor="middle">Consumer</text> <text x="55" y="38" font-family="sans-serif" font-size="9" fill="#581C87" text-anchor="middle">B -> C -> A 순 수신</text> </g> </svg> </div>


서로 다른 우선순위로 메시지 3개를 발행합니다. 소비 시 우선순위가 높은 순서대로 처리됩니다.

In [ ]:
priorities = [
    (1, "low-priority task"),
    (10, "CRITICAL task"),
    (5, "medium-priority task"),
]
for prio, desc in priorities:
    resp = await client.post(
        "/rabbitmq/priority/publish",
        params={"queue_name": "task-queue"},
        json={"content": desc, "priority": prio}
    )
    print(f"Priority {prio:>2}: {resp.json()}")

큐 상태를 조회하여 메시지가 쌓였는지 확인합니다.

In [ ]:
resp = await client.get("/rabbitmq/queue/info/task-queue")
print(f"Task queue info: {resp.json()}")

### Priority Queue 소비 순서

Consumer가 큐에서 메시지를 가져올 때, RabbitMQ는 내부적으로 우선순위별 서브큐를 유지합니다.  
위 예제에서 소비 순서는: `CRITICAL(10)` → `medium(5)` → `low(1)` 이 됩니다.

> **참고**: 메시지가 이미 Consumer에게 전달(prefetch)된 상태라면 우선순위 재정렬이 발생하지 않습니다.

---
## 9. Pattern 6: TTL (Time-To-Live)

### 음식 유통기한으로 이해하기

TTL은 **음식의 유통기한**과 같습니다.

마트에서 파는 음식에는 유통기한이 있습니다.
유통기한이 지나면 **자동으로 폐기**됩니다.
RabbitMQ 메시지도 마찬가지로, TTL을 설정하면 지정 시간이 지난 메시지가 자동으로 삭제됩니다.


<div align="center"> <svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 800 200" width="100%" height="auto"> <rect width="100%" height="100%" fill="#F8FAFC" rx="12" /> <text x="400" y="30" font-family="-apple-system, sans-serif" font-size="14" font-weight="bold" fill="#0F172A" text-anchor="middle">Pattern 6: TTL (Time-To-Live) 유통기한 및 자동 소멸</text> <g transform="translate(80, 70)"> <rect x="0" y="0" width="460" height="80" rx="8" fill="#F1F5F9" stroke="#475569" stroke-width="2" /> <text x="20" y="-10" font-family="-apple-system, sans-serif" font-size="11" font-weight="bold" fill="#475569">Queue (유통기한 만료대기)</text> <g transform="translate(20, 20)"> <rect x="0" y="0" width="110" height="40" rx="4" fill="#DCFCE7" stroke="#15803D" /> <text x="55" y="20" font-family="monospace" font-size="9" fill="#14532D" text-anchor="middle">Msg 1 (TTL=5s)</text> <text x="55" y="32" font-family="-apple-system, sans-serif" font-size="8" fill="#166534" text-anchor="middle">대기 2초 남음</text> </g> <g transform="translate(150, 20)"> <rect x="0" y="0" width="110" height="40" rx="4" fill="#FEE2E2" stroke="#B91C1C" stroke-dasharray="3 3" /> <text x="55" y="20" font-family="monospace" font-size="9" fill="#991B1B" text-anchor="middle">Msg 2 (TTL=5s)</text> <text x="55" y="32" font-family="-apple-system, sans-serif" font-size="8" fill="#991B1B" text-anchor="middle">시간만료 (DROP!)</text> </g> </g> <path d="M 330 150 L 330 170 L 610 170 L 610 135" fill="none" stroke="#EF4444" stroke-width="2" stroke-dasharray="4 2" /><polygon points="613,141 610,135 607,141" fill="#EF4444" /> <g transform="translate(560, 75)"> <rect x="0" y="0" width="160" height="50" rx="6" fill="#FEE2E2" stroke="#B91C1C" stroke-width="1.5" /> <text x="80" y="24" font-family="-apple-system, sans-serif" font-size="11" font-weight="bold" fill="#991B1B" text-anchor="middle">Dead Letter (DLX)</text> <text x="80" y="38" font-family="-apple-system, sans-serif" font-size="9" fill="#7F1D1D" text-anchor="middle">자동 폐기 또는 DLQ 이송</text> </g> </svg> </div>


TTL이 짧은(5초) 메시지를 발행합니다. 5초 후에 큐에서 자동 삭제됩니다.

In [ ]:
resp = await client.post(
    "/rabbitmq/ttl/publish",
    params={"queue_name": "temp-queue"},
    json={"content": "expires in 5 seconds", "ttl_ms": 5000}
)
print(f"TTL publish (5s): {resp.json()}")

발행 직후 큐 상태를 확인하면 메시지가 존재합니다.

In [ ]:
resp = await client.get("/rabbitmq/queue/info/temp-queue")
print(f"즉시 조회: {resp.json()}")

6초 후에 다시 조회하면 TTL이 만료되어 메시지가 사라진 것을 확인할 수 있습니다.

In [ ]:
import asyncio

print("6초 대기 중...")
await asyncio.sleep(6)

resp = await client.get("/rabbitmq/queue/info/temp-queue")
print(f"6초 후 조회: {resp.json()}")

### TTL + DLQ 조합

TTL이 만료된 메시지를 DLQ로 보내도록 설정하면, 만료된 메시지도 추적할 수 있습니다.  
이는 `x-dead-letter-exchange`와 `expiration`을 함께 사용하여 구현합니다.

```text
[만료된 메시지] ──▶ DLX ──▶ DLQ (만료 메시지 보관)
```


---
## 10. RabbitMQ Management UI 활용법

RabbitMQ는 웹 기반 관리 UI를 제공합니다.

| 항목 | 값 |
|------|----|
| URL | http://localhost:15672 |
| Username | `guest` |
| Password | `guest` |

### 주요 메뉴

```text
┌─────────────────────────────────────────────────────────┐
│  RabbitMQ Management                                    │
├──────────┬──────────┬──────────┬──────────┬────────────┤
│ Overview │ Connect- │ Channels │ Exchanges│   Queues   │
│          │ ions     │          │          │            │
├──────────┴──────────┴──────────┴──────────┴────────────┤
│                                                         │
│  [Overview]   - 전체 메시지 비율, 노드 상태              │
│  [Queues]     - 큐별 메시지 수, 소비자 수, 메시지 비율   │
│  [Exchanges]  - Exchange 목록, 타입, 바인딩 정보         │
│  [Connections]- 연결된 클라이언트 목록                    │
│  [Channels]   - 채널별 prefetch, unacked 수             │
│                                                         │
└─────────────────────────────────────────────────────────┘
```

### 실습에서 확인할 것들

1. **Queues** 탭에서 `order-queue`, `task-queue`, `payment-queue.dlq` 등 확인
2. **Exchanges** 탭에서 `events-exchange`(fanout), `order-events`(topic) 확인
3. 각 큐를 클릭하면 **Get Messages** 버튼으로 메시지 내용을 직접 확인 가능

---
## 11. 정리 및 요약

### 6가지 패턴 비교 테이블

| 패턴 | Exchange 타입 | Routing Key | 사용 사례 | 메서드 |
|------|-------------|-------------|-----------|--------|
| **Direct Queue** | Default (`""`) | 큐 이름 | 1:1 작업 분배 | `publish()` |
| **Fanout** | FANOUT | 무시됨 | 브로드캐스트, 알림 | `publish_fanout()` |
| **Topic** | TOPIC | 패턴 (`*`, `#`) | 선택적 라우팅 | `publish_topic()` |
| **DLQ** | FANOUT (DLX) | - | 실패 메시지 격리 | `setup_dlq()` |
| **Priority** | Default (`""`) | 큐 이름 | 긴급 작업 우선 처리 | `publish_priority()` |
| **TTL** | Default (`""`) | 큐 이름 | 임시 데이터, 캐시 | `publish_ttl()` |

### API 엔드포인트 요약

| 메서드 | 엔드포인트 | 패턴 |
|--------|-----------|------|
| POST | `/rabbitmq/direct/publish` | Direct |
| POST | `/rabbitmq/fanout/publish` | Fanout |
| POST | `/rabbitmq/fanout/bind` | Fanout 바인딩 |
| POST | `/rabbitmq/topic/publish` | Topic |
| POST | `/rabbitmq/topic/bind` | Topic 바인딩 |
| POST | `/rabbitmq/dlq/setup` | DLQ 구성 |
| GET  | `/rabbitmq/dlq/messages` | DLQ 조회 |
| POST | `/rabbitmq/priority/publish` | Priority |
| POST | `/rabbitmq/ttl/publish` | TTL |
| GET  | `/rabbitmq/queue/info/{queue}` | 큐 정보 |

### 핵심 설계 원칙

1. **내구성(Durability)**: `durable=True` + `DeliveryMode.PERSISTENT`로 메시지 유실 방지
2. **흐름 제어(Flow Control)**: `prefetch_count=10`으로 Consumer 과부하 방지
3. **자동 복구**: `connect_robust()`로 네트워크 장애 시 자동 재연결
4. **실패 격리**: DLQ 패턴으로 실패한 메시지를 별도 관리

### RabbitMQ 4 — 무엇이 달라졌나?

이 프로젝트는 **RabbitMQ 4**를 사용합니다. 3.x 대비 주요 변경사항:

| 변경사항 | 설명 |
|----------|------|
| **Quorum Queue 기본화** | Raft 합의 기반 복제 큐가 표준. Classic Mirrored Queue 제거 |
| **Classic Queue 유지** | 비복제 Classic Queue는 여전히 사용 가능 (이 프로젝트에서 사용하는 방식) |
| **기본 재전송 제한** | Quorum Queue의 기본 delivery limit = 20 (무한 재시도 방지) |
| **AMQP 1.0 네이티브** | AMQP 1.0 프로토콜 직접 지원 (0-9-1도 계속 지원) |

```text
[큐 타입 선택 가이드 - RabbitMQ 4]

  데이터 안전이 중요한가? ──→ Yes ──→ Quorum Queue (Raft 복제)
           │                              • 3+ 노드 클러스터에서 사용
           │                              • 기본 delivery limit = 20
           │
           └──→ No ──→ Classic Queue (단일 노드)
                         • 개발/학습 환경에 적합
                         • 이 프로젝트에서 사용하는 방식
```

> **왜 이 프로젝트에서는 Classic Queue를 사용하는가?**
> 단일 노드 학습 환경에서는 Quorum Queue의 복제 이점이 없습니다.
> 프로덕션에서 RabbitMQ 클러스터를 운영한다면 Quorum Queue를 기본으로 사용하세요.

마지막으로 httpx 클라이언트를 정리합니다.

In [ ]:
await client.aclose()
print("httpx client closed.")